In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = "gpt-oss-20b"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1.1. Define SHOP_TOOLS_SCHEMA — tool descriptions for the LLM.
#
# Below are stub functions with signatures but no descriptions.
# The LLM needs to understand what each tool does and what its parameters mean.
#
# Task: add a docstring (description + Args) to each function.
# The convert_to_openai_tool() function from the template will generate the JSON schema automatically.
# For docstring format details, see Google-style docstrings.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    # YOUR CODE HERE: add a docstring describing the tool and its parameters
    """
    Searh for products in the catalog.
    
    Args:
        query: Search query string(matches against name, category, brand, and tags)
        category: Filter by product category. Options: "headphoness", "earbuds", "mouse", "keyboard", "ereader"
        brand: Filter by brand name (case-insensitive)
        max_price: Maximum price filter
        sort_by: Sort results. Options: "price_asc" (lowest first), "rating_desc" (highest first)

    """
    return TOOLS.search_products(query, category, brand, max_price, sort_by)

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    # YOUR CODE HERE: add a docstring
    """
    Add a product to the shopping cart.

    Args:
        product_id: Product identifier (e.g., "p1", "p2", etc.)
        quantity: Number of items to add (default: 1)
    """
    return TOOLS.add_to_cart(ShopState(), product_id, quantity)

# YOUR CODE HERE: generate the schema
SHOP_TOOLS_SCHEMA = [
     convert_to_openai_tool(search_products),
     convert_to_openai_tool(add_to_cart),
]


# 1.2. Implement run_shopping_agent — a ReAct shop agent.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    # YOUR CODE HERE
    messages = [
        SystemMessage(content="You are a helpful shopping assistant. Use the avail able tools to search for products and add them to the cart. When showing search results, include product names, prices, and ratings. When the user asks to add somet hing to cart, use add_to_cart with the correct product_id."),
        HumanMessage(content=user_message)
    ]

    while True:
        # Get response from llm
        response = llm_chat(messages, SHOP_TOOLS_SCHEMA)

        # Check if there are tool calls
        if not response.tool_calls:
            return response.content

        # Process each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]
            
        # Execute the appropriate tool
            if tool_name == "search_products":
                result = tools.search_products(**tool_args)
                state.last_results = result
                tracer_str = json.dumps(result, ensure_ascii=False)[:1000]
            elif tool_name == "add_to_cart":
                result = tools.add_to_cart(state, **tool_args)
                tracer.record(tool_name, tool_args, result)
                result_str = json.dumps(result, ensure_ascii=False)
            else:
                result_str = f"Unknown tool: {tool_name}"
                
          # Add the assistant's message and tool result to history
            messages.append(response)
            messages.append(ToolMessage(content=result_str, tool_call_id=tool_id))
        
    #raise NotImplementedError


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Recommended profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
     # YOUR CODE HERE
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}
        #raise NotImplementedError

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(profile, f, ensure_ascii=False, indent=2)

            #raise NotImplementedError

def update_profile(key: str, value: str) -> dict:
    """Update user profile with a preference.

    Args:
        key: Profile field to update. Options: "name", "brand", "max_price", "color", "category"
        value: Value to save
    """
    profile = load_profile()
    profile[key] = value
    save_profile(profile)
    return {"ok": True, "profile": profile}

SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
    convert_to_openai_tool(update_profile),
]

def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """
    
    profile = load_profile(profile_path)
    profile_text = ""
    if profile:
        profile_text = "\nUser profile:\n"
        for key, value in profile.items():
            profile_text += f"  - {key}: {value}\n"
        profile_text += "\nUse these preferences when searching and recommending products.\n"
    system_message = SystemMessage(
        content=f"You are a helpful shopping assistant that remembers user preferences.{profile_text}Use the update_profile tool when the user mentions their preferences for the first time. When showing search results, include product names, prices, and ratings."
    )
    messages = [system_message] + history + [HumanMessage(content=user_message)]
    while True:
        response = llm_chat(messages, SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(response)
        history.append(response)
        if not response.tool_calls:
            return response.content, history
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]
            if tool_name == "search_products":
                result = tools.search_products(**tool_args)
                state.last_results = result
                tracer.record(tool_name, tool_args, result)
                result_str = json.dumps(result, ensure_ascii=False)[:1000]
            elif tool_name == "add_to_cart":
                result = tools.add_to_cart(state, **tool_args)
                tracer.record(tool_name, tool_args, result)
                result_str = json.dumps(result, ensure_ascii=False)
            elif tool_name == "update_profile":
                result = update_profile(**tool_args)
                tracer.record(tool_name, tool_args, result)
                result_str = json.dumps(result, ensure_ascii=False)
                profile = load_profile(profile_path)
                profile_text = ""
                if profile:
                    profile_text = "\nUser profile:\n"
                    for key, value in profile.items():
                        profile_text += f"  - {key}: {value}\n"
                system_message.content = f"You are a helpful shopping assistant that remembers user preferences.{profile_text}Use the update_profile tool when the user mentions their preferences for the first time. When showing search results, include product names, prices, and ratings."
                messages[0] = system_message
            else:
                result_str = f"Unknown tool: {tool_name}"
                
            tool_message = ToolMessage(content=result_str, tool_call_id=tool_id)
            messages.append(tool_message)
            history.append(tool_message)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Implement a system of four agents + an orchestrator.
# Goal — find the best product and honestly describe its pros and cons.
# Agents work in a chain via a shared AgentContext object (defined in the template).
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   No CartAgent needed — if the user asks to add to cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        # YOUR CODE HERE
        #raise NotImplementedError
        messages = [
            SystemMessage(content="You are a product search specialist. Your task is to search for products based on the user's query. Use the search_products tool to find relevant products. Return up to 5 best matches."),
            HumanMessage(content=f"Find products for: {ctx.query}")
        ]
        
        # Add max_price to search if specified
        if ctx.max_price:
            messages.append(HumanMessage(content=f"Filter by max price: ${ctx.max_price}"))
        
        # Call LLM with search tool only
        search_tools = [convert_to_openai_tool(search_products)]
        response = llm_chat(messages, search_tools)
        
        # Process tool calls
        if response.tool_calls:
            for tool_call in response.tool_calls:
                if tool_call["name"] == "search_products":
                    result = tools.search_products(**tool_call["args"])
                    ctx.candidates = result[:5]  
                    tracer.record("retriever_search", tool_call["args"], result)
                    
                    # Extract max_price from search if present
                    if "max_price" in tool_call["args"]:
                        ctx.max_price = tool_call["args"]["max_price"]
        
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        # YOUR CODE HERE
        #raise NotImplementedError
        if not ctx.candidates:
            return ctx
        
        pros_analysis = {}
        for product in ctx.candidates:
            messages = [
                SystemMessage(content="You are a product analyst. Write 1-2 sentences highlighting the key advantages of this product. Be specific and factual."),
                HumanMessage(content=f"Product: {product['name']}\nBrand: {product['brand']}\nPrice: ${product['price']}\nRating: {product['rating']}\nTags: {', '.join(product['tags'])}\n\nWhat are the pros of this product?")
            ]
            
            response = llm_chat(messages)
            pros_analysis[product['id']] = response.content
            tracer.record("analyze_pros", {"product_id": product['id'], "product_name": product['name']}, response.content)
        
        ctx.pros = pros_analysis
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        # YOUR CODE HERE
        #raise NotImplementedError
        if not ctx.candidates:
            return ctx
        
        cons_analysis = {}
        for product in ctx.candidates:
            messages = [
                SystemMessage(content="You are a product analyst. Write 1-2 sentences highlighting potential drawbacks or limitations of this product. Be honest and constructive."),
                HumanMessage(content=f"Product: {product['name']}\nBrand: {product['brand']}\nPrice: ${product['price']}\nRating: {product['rating']}\nTags: {', '.join(product['tags'])}\n\nWhat are the cons of this product?")
            ]
            
            response = llm_chat(messages)
            cons_analysis[product['id']] = response.content
            tracer.record("analyze_cons", {"product_id": product['id'], "product_name": product['name']}, response.content)
        
        ctx.cons = cons_analysis
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        # YOUR CODE HERE
        #raise NotImplementedError
        if not ctx.candidates:
            return ctx
        
        # Filter by max_price if set
        candidates = ctx.candidates
        if ctx.max_price:
            candidates = [p for p in candidates if p["price"] <= ctx.max_price]
        
        if not candidates:
            ctx.best = None
            tracer.record("rank_candidates", {"error": "No products within price range"}, None)
            return ctx
        
        # Sort by rating descending, then price ascending
        best_product = max(candidates, key=lambda x: (x["rating"], -x["price"]))
        ctx.best = best_product
        
        tracer.record("rank_candidates", {
            "candidates_count": len(candidates),
            "best_product_id": best_product["id"],
            "best_product_name": best_product["name"],
            "best_rating": best_product["rating"],
            "best_price": best_product["price"]
        }, best_product)
        
        return ctx

class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        # YOUR CODE HERE
        # ##raise NotImplementedError
        
        trace = []
        ctx = AgentContext(query=user_message)
        
        # Step 1: Retriever
        trace.append({"delegate_retriever": "searching for products"})
        ctx = self.retriever.run(ctx, state, tools, ToolTracer())
        
        # Step 2: Pros analysis
        if ctx.candidates:
            trace.append({"delegate_pros": f"analyzing {len(ctx.candidates)} products"})
            ctx = self.pros_agent.run(ctx, ToolTracer())
        
        # Step 3: Cons analysis
        if ctx.candidates:
            trace.append({"delegate_cons": f"analyzing {len(ctx.candidates)} products"})
            ctx = self.cons_agent.run(ctx, ToolTracer())
        
        # Step 4: Ranking
        trace.append({"delegate_ranker": "selecting best product"})
        ctx = self.ranker.run(ctx, ToolTracer())
        
        # Generate response
        if ctx.best:
            best = ctx.best
            pros_text = ctx.pros.get(best['id'], "No pros analysis available")
            cons_text = ctx.cons.get(best['id'], "No cons analysis available")
            
            response = f"""**Best Recommendation: {best['name']}**

**Price:** ${best['price']}
**Rating:** {best['rating']}/5.0
**Brand:** {best['brand']}
**Category:** {best['category']}
**Tags:** {', '.join(best['tags'])}

**Pros:**
{pros_text}

**Cons:**
{cons_text}

This product best matches your requirements based on price, rating, and features."""
            
            # Check if user wants to add to cart
            if "add to cart" in user_message.lower() or "buy" in user_message.lower():
                trace.append({"delegate_cart": "adding to cart"})
                result = tools.add_to_cart(state, best['id'])
                ctx.cart_result = result
                response += f"\n\n**Cart Status:** Added to cart successfully!"
        else:
            response = "I couldn't find any products matching your criteria. Please try a different search."
        
        return AgentResult(response=response, trace=trace, context=ctx)



In [ ]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


In [ ]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")

In [ ]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")
